<a href="https://colab.research.google.com/github/detektor777/colab_list_image/blob/main/supir.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title ##**Install** { display-mode: "form" }
%%capture
!rm -rf SUPIR
# Clear cache from previous attempts
!rm -rf ~/.cache/huggingface/hub/models--laion* ~/.cache/clip ~/.cache/torch/hub/checkpoints

# Clone the official SUPIR repository
!git clone https://github.com/Fanghua-Yu/SUPIR.git
%cd SUPIR

# Install only the needed packages directly, pinning the correct open-clip-torch==2.24.0 version
!pip install -q pytorch-lightning omegaconf einops timm open-clip-torch==2.24.0 k-diffusion bitsandbytes transformers accelerate diffusers scikit-image huggingface_hub

import os
import re
from huggingface_hub import hf_hub_download, snapshot_download

os.makedirs("checkpoints", exist_ok=True)

# DOWNLOAD THE PHOTOREALISTIC BASE JUGGERNAUT-XL V9
print("Downloading the photorealistic base Juggernaut-XL v9 (~6.6 GB)...")
sdxl_path = hf_hub_download(
    repo_id="RunDiffusion/Juggernaut-XL-v9",
    filename="Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors",
    local_dir="checkpoints"
)

# DOWNLOAD SUPIR-v0Q WEIGHTS
print("Downloading SUPIR-v0Q weights (~5.3 GB)...")
supir_q_path = hf_hub_download(
    repo_id="camenduru/SUPIR",
    filename="SUPIR-v0Q.ckpt",
    local_dir="checkpoints"
)

# DOWNLOAD SUPIR-v0F WEIGHTS (FOR FIDELITY MODE)
print("Downloading SUPIR-v0F weights (~5.3 GB)...")
supir_f_path = hf_hub_download(
    repo_id="camenduru/SUPIR",
    filename="SUPIR-v0F.ckpt",
    local_dir="checkpoints"
)

print("Downloading LLaVA-v1.5-7B (~13 GB, once, then cached)...")
llava_path = snapshot_download(
    repo_id="liuhaotian/llava-v1.5-7b",
    local_dir="/root/llava-v1.5-7b",
)

print("Weights downloaded:", sdxl_path, supir_q_path, supir_f_path, llava_path)

# Patch 1: CKPT_PTH.py
ckpt_pth_content = f'''
LLAVA_CLIP_PATH = None
LLAVA_MODEL_PATH = "{llava_path}"
SDXL_CLIP1_PATH = None
SDXL_CLIP2_CKPT_PTH = None
'''
with open("CKPT_PTH.py", "w") as f:
    f.write(ckpt_pth_content)

# Patch 2: options/SUPIR_v0.yaml (now configure paths to both SUPIR versions)
with open("options/SUPIR_v0.yaml", "r") as f:
    cfg = f.read()
cfg = cfg.replace("\r\n", "\n")
cfg = re.sub(r"SDXL_CKPT:.*", f"SDXL_CKPT: {sdxl_path}", cfg)
cfg = re.sub(r"SUPIR_CKPT_Q:.*", f"SUPIR_CKPT_Q: {supir_q_path}", cfg)
cfg = re.sub(r"SUPIR_CKPT_F:.*", f"SUPIR_CKPT_F: {supir_f_path}", cfg)
with open("options/SUPIR_v0.yaml", "w") as f:
    f.write(cfg)

# Patch 2b: llava/model/language_model/llava_llama.py — conflict registering
# the "llava" model type with newer transformers versions
llava_llama_path = os.path.join("llava", "model", "language_model", "llava_llama.py")
if os.path.exists(llava_llama_path):
    with open(llava_llama_path, "r") as f:
        llava_llama_content = f.read()
    llava_llama_content = llava_llama_content.replace("\r\n", "\n")

    old_register = 'AutoConfig.register("llava", LlavaConfig)'
    new_register = 'AutoConfig.register("llava", LlavaConfig, exist_ok=True)'
    if old_register in llava_llama_content:
        llava_llama_content = llava_llama_content.replace(old_register, new_register)
        print("llava_llama.py: 'llava' registration conflict fixed (exist_ok=True).")
    else:
        print("WARNING: AutoConfig registration line not found.")

    with open(llava_llama_path, "w") as f:
        f.write(llava_llama_content)
else:
    print("WARNING: llava_llama.py file not found at the expected path.")

# Patch 2c: llava/model/__init__.py — remove the import of the MPT variant of LLaVA
# (not needed for llava-v1.5-7b; its code pulls in the _expand_mask function
# that was removed in newer transformers, which breaks the whole llava package
# import before the weights are even loaded)
llava_init_path = os.path.join("llava", "model", "__init__.py")
with open(llava_init_path, "r") as f:
    llava_init_content = f.read()

lines = llava_init_content.splitlines()
new_lines = [ln for ln in lines if "mpt" not in ln.lower()]
llava_init_content_new = "\n".join(new_lines) + "\n"

with open(llava_init_path, "w") as f:
    f.write(llava_init_content_new)

print("llava/model/__init__.py: MPT variant import removed.")

# Patch 2d: llava/model/builder.py — newer transformers versions don't accept
# load_in_8bit directly in the model's __init__, it must be wrapped in BitsAndBytesConfig
builder_path = os.path.join("llava", "model", "builder.py")
with open(builder_path, "r") as f:
    builder_content = f.read()
builder_content = builder_content.replace("\r\n", "\n")

old_8bit_block = "    if load_8bit:\n        kwargs['load_in_8bit'] = True"
new_8bit_block = (
    "    if load_8bit:\n"
    "        from transformers import BitsAndBytesConfig\n"
    "        kwargs['quantization_config'] = BitsAndBytesConfig(load_in_8bit=True)"
)
if old_8bit_block in builder_content:
    builder_content = builder_content.replace(old_8bit_block, new_8bit_block)
    print("builder.py: load_in_8bit replaced with BitsAndBytesConfig(load_in_8bit=True).")
else:
    builder_content = builder_content.replace(
        "kwargs['load_in_8bit'] = True",
        "kwargs['quantization_config'] = __import__('transformers').BitsAndBytesConfig(load_in_8bit=True)"
    )
    print("builder.py: load_in_8bit replaced (fallback method).")

old_4bit_line = "kwargs['load_in_4bit'] = True"
if old_4bit_line in builder_content:
    builder_content = builder_content.replace(
        old_4bit_line,
        "kwargs['quantization_config'] = __import__('transformers').BitsAndBytesConfig(load_in_4bit=True)"
    )
    print("builder.py: load_in_4bit replaced with BitsAndBytesConfig(load_in_4bit=True).")

with open(builder_path, "w") as f:
    f.write(builder_content)

# Patch 2e: llava/model/llava_arch.py — in newer transformers past_key_values
# is a DynamicCache object, not a tuple, so [-1][-1] indexing doesn't work.
llava_arch_path = os.path.join("llava", "model", "llava_arch.py")
with open(llava_arch_path, "r") as f:
    llava_arch_content = f.read()
llava_arch_content = llava_arch_content.replace("\r\n", "\n")

old_expr = "past_key_values[-1][-1].shape[-2] + 1"
new_expr = (
    "(past_key_values.get_seq_length() if hasattr(past_key_values, 'get_seq_length') "
    "else past_key_values[-1][-1].shape[-2]) + 1"
)

if old_expr in llava_arch_content:
    llava_arch_content = llava_arch_content.replace(old_expr, new_expr)
    print("llava_arch.py: past_key_values indexing replaced with a version compatible with DynamicCache.")
else:
    print("WARNING: the target expression in llava_arch.py was not found — please check manually.")

with open(llava_arch_path, "w") as f:
    f.write(llava_arch_content)

# Patch 3: test.py — safe LLaVA import + detailed traceback on error
with open("test.py", "r") as f:
    content = f.read()
content = content.replace("\r\n", "\n")

content = content.replace(
    "from llava.llava_agent import LLavaAgent",
    "try:\n    from llava.llava_agent import LLavaAgent\nexcept Exception as e:\n    import traceback\n    print('LLaVA skipped (import error):')\n    traceback.print_exc()\n    LLavaAgent = None"
)

old_line = "    llava_agent = LLavaAgent(LLAVA_MODEL_PATH, device=LLaVA_device, load_8bit=args.load_8bit_llava, load_4bit=False)"
new_line = (
    "    if LLavaAgent is not None:\n"
    "        try:\n"
    "            llava_agent = LLavaAgent(LLAVA_MODEL_PATH, device=LLaVA_device, load_8bit=args.load_8bit_llava, load_4bit=False)\n"
    "        except Exception as e:\n"
    "            import traceback\n"
    "            print('LLaVA skipped (agent creation error):')\n"
    "            traceback.print_exc()\n"
    "            llava_agent = None\n"
    "    else:\n"
    "        llava_agent = None"
)
if old_line in content:
    content = content.replace(old_line, new_line)
    print("test.py: LLavaAgent creation wrapped in try/except.")
else:
    print("WARNING: LLavaAgent creation line not found — please check manually.")

# Patch 4: test.py — GUARANTEED cleanup of LLaVA from VRAM (full object deletion)
# FIXED indentation for Else (shifted left by 4 spaces)
old_captions_block = (
    "        if use_llava:\n"
    "            captions = llava_agent.gen_image_caption([clean_PIL_img])\n"
    "        else:\n"
    "            captions = ['']"
)

new_captions_block = (
    "        if use_llava and llava_agent is not None:\n"
    "            try:\n"
    "                captions = llava_agent.gen_image_caption([clean_PIL_img])\n"
    "                print(f'>>> LLaVA CAPTION: {captions}')\n"
    "                # Fully remove LLaVA from VRAM by clearing references\n"
    "                if hasattr(llava_agent, \"model\"):\n"
    "                    del llava_agent.model\n"
    "                del llava_agent\n"
    "                llava_agent = None\n"
    "                import torch, gc\n"
    "                gc.collect()\n"
    "                torch.cuda.empty_cache()\n"
    "                print('[CLEANUP]: LLaVA fully removed from VRAM before the generation step.')\n"
    "            except Exception as e:\n"
    "                print(f'LLaVA error: {e}')\n"
    "                captions = ['']\n"
    "        else:\n"
    "            captions = ['']"
)

if old_captions_block in content:
    content = content.replace(old_captions_block, new_captions_block)
    print("test.py: added patch to fully remove LLaVA from VRAM.")
else:
    old_captions_block_alt = (
        "    if use_llava:\n"
        "        captions = llava_agent.gen_image_caption([clean_PIL_img])\n"
        "    else:\n"
        "        captions = ['']"
    )
    new_captions_block_alt = (
        "    if use_llava and llava_agent is not None:\n"
        "        try:\n"
        "            captions = llava_agent.gen_image_caption([clean_PIL_img])\n"
        "            print(f'>>> LLaVA CAPTION: {captions}')\n"
        "            if hasattr(llava_agent, \"model\"):\n"
        "                del llava_agent.model\n"
        "            del llava_agent\n"
        "            llava_agent = None\n"
        "            import torch, gc\n"
        "            gc.collect()\n"
        "            torch.cuda.empty_cache()\n"
        "            print('[CLEANUP]: LLaVA fully removed from VRAM before the generation step (alt).')\n"
        "        except Exception as e:\n"
        "            print(f'LLaVA error: {e}')\n"
        "            captions = ['']\n"
        "    else:\n"
        "        captions = ['']"
    )
    if old_captions_block_alt in content:
        content = content.replace(old_captions_block_alt, new_captions_block_alt)
        print("test.py: added patch to fully remove LLaVA from VRAM (alt).")
    else:
        print("WARNING: could not find the captions block to patch for VRAM cleanup.")

with open("test.py", "w") as f:
    f.write(content)

# Patch 5: SUPIR/utils/tilevae.py
tilevae_path = os.path.join("SUPIR", "utils", "tilevae.py")
with open(tilevae_path, "r") as f:
    tilevae_content = f.read()
tilevae_content = tilevae_content.replace("\r\n", "\n")

old_block = '''        if is_xformers_available:
            # task_queue.append(('attn', lambda x, net=net: attn_forward_new_xformers(net, x)))
            task_queue.append(
                ('attn', lambda x, net=net: xformer_attn_forward(net, x)))
        elif hasattr(F, "scaled_dot_product_attention"):
            task_queue.append(('attn', lambda x, net=net: attn_forward_new_pt2_0(net, x)))
        else:
            task_queue.append(('attn', lambda x, net=net: attn_forward_new(net, x)))'''

new_block = '''        task_queue.append(('attn', lambda x, net=net: attn_forward(net, x)))'''

if old_block in tilevae_content:
    tilevae_content = tilevae_content.replace(old_block, new_block)
    print("tilevae.py: attention selection branch replaced with attn_forward.")
else:
    print("WARNING: target block in tilevae.py not found.")

with open(tilevae_path, "w") as f:
    f.write(tilevae_content)

# Patch 5b: sgm/modules/diffusionmodules/model.py — if xformers is unavailable,
# redirect attn_type "vanilla-xformers" to the standard "vanilla" (AttnBlock),
# which can work via native scaled_dot_product_attention without xformers.
diff_model_path = os.path.join("sgm", "modules", "diffusionmodules", "model.py")
with open(diff_model_path, "r") as f:
    diff_model_content = f.read()
diff_model_content = diff_model_content.replace("\r\n", "\n")

old_func_def = 'def make_attn(in_channels, attn_type="vanilla", attn_kwargs=None):'
new_func_def = (
    'def make_attn(in_channels, attn_type="vanilla", attn_kwargs=None):\n'
    '    if attn_type == "vanilla-xformers" and not XFORMERS_IS_AVAILABLE:\n'
    '        attn_type = "vanilla"'
)

if old_func_def in diff_model_content:
    diff_model_content = diff_model_content.replace(old_func_def, new_func_def)
    print("model.py: make_attn now falls back to AttnBlock if xformers is unavailable.")
else:
    old_func_def_single = "def make_attn(in_channels, attn_type='vanilla', attn_kwargs=None):"
    if old_func_def_single in diff_model_content:
        diff_model_content = diff_model_content.replace(old_func_def_single, new_func_def)
        print("model.py: make_attn now falls back to AttnBlock if xformers is unavailable (single quotes).")
    else:
        print("WARNING: make_attn function not found in model.py — please check manually.")

with open(diff_model_path, "w") as f:
    f.write(diff_model_content)

# Patch 6: sgm/modules/encoders/modules.py — DISABLE DOWNLOADING 10 GB OF OpenCLIP
# (the model is initialized directly on GPU, without loading CPU RAM)
modules_path = os.path.join("sgm", "modules", "encoders", "modules.py")
with open(modules_path, "r") as f:
    modules_content = f.read()
modules_content = modules_content.replace("\r\n", "\n")

old_call = '''        model, _, _ = open_clip.create_model_and_transforms(
            arch,
            device=torch.device("cpu"),
            pretrained=version if SDXL_CLIP2_CKPT_PTH is None else SDXL_CLIP2_CKPT_PTH,
        )
        del model.visual'''

new_call = '''        model, _, _ = open_clip.create_model_and_transforms(
            arch,
            device=torch.device("cuda"),
            pretrained="" if SDXL_CLIP2_CKPT_PTH is None else SDXL_CLIP2_CKPT_PTH,
            precision="fp16",
        )
        model = model.half()
        del model.visual'''

if old_call in modules_content:
    modules_content = modules_content.replace(old_call, new_call)
    print("modules.py: FrozenOpenCLIPEmbedder2 now loads onto GPU (cuda) in fp16.")
else:
    print("WARNING: target block in modules.py not found.")

# Remove the forced upcast to float32, which was eating up all the VRAM
modules_content = modules_content.replace(
    "model = model.float()",
    "model = model.half()"
)

with open(modules_path, "w") as f:
    f.write(modules_content)

# Patch 7: SUPIR/util.py — create the model directly on GPU (cuda) in the compact float16 format
# This COMPLETELY eliminates CPU RAM allocation (reduces down to 7.2 GB VRAM).
# We keep checkpoint loading on CPU ('location=cpu') for optimal memory balance.
util_path = os.path.join("SUPIR", "util.py")
with open(util_path, "r") as f:
    util_content = f.read()
util_content = util_content.replace("\r\n", "\n")

# 1. Force all tensors in load_state_dict to be cast to float16 (half) on CPU
# This fixes the dtype mismatch issue and cuts CPU RAM usage by 50% when reading checkpoints!
old_load_state_dict = """def load_state_dict(ckpt_path, location='cpu'):
    if ckpt_path.endswith('.safetensors'):
        state_dict = load_safetensors(ckpt_path, device=location)
    else:
        state_dict = get_state_dict(torch.load(ckpt_path, map_location=torch.device(location)))
    return state_dict"""

new_load_state_dict = """def load_state_dict(ckpt_path, location='cpu'):
    if ckpt_path.endswith('.safetensors'):
        state_dict = load_safetensors(ckpt_path, device=location)
    else:
        state_dict = get_state_dict(torch.load(ckpt_path, map_location=torch.device(location)))
    # Cast weights to float16 on CPU to save RAM and match dtypes
    for k in list(state_dict.keys()):
        if hasattr(state_dict[k], 'half'):
            state_dict[k] = state_dict[k].half()
    return state_dict"""

if old_load_state_dict in util_content:
    util_content = util_content.replace(old_load_state_dict, new_load_state_dict)
    print("util.py: Forced float16 weight casting on CPU has been added.")
else:
    # Double-quote variant
    old_load_state_dict_double = """def load_state_dict(ckpt_path, location="cpu"):
    if ckpt_path.endswith('.safetensors'):
        state_dict = load_safetensors(ckpt_path, device=location)
    else:
        state_dict = get_state_dict(torch.load(ckpt_path, map_location=torch.device(location)))
    return state_dict"""
    if old_load_state_dict_double in util_content:
        util_content = util_content.replace(old_load_state_dict_double, new_load_state_dict)
        print("util.py: Forced float16 weight casting on CPU has been added (double quotes).")
    else:
        print("WARNING: could not inject weight casting into load_state_dict.")

# 2. Wrap model instantiation in a 'with torch.device("cuda")' context and float16
util_content = util_content.replace(
    "model = instantiate_from_config(config.model).cpu()",
    "import torch\n    old_dtype = torch.get_default_dtype()\n    torch.set_default_dtype(torch.float16)\n    try:\n        with torch.device('cuda'):\n            model = instantiate_from_config(config.model)\n    finally:\n        torch.set_default_dtype(old_dtype)"
)

# 3. Clear memory after loading SDXL (8-space indent)
util_content = util_content.replace(
    "model.load_state_dict(load_state_dict(config.SDXL_CKPT), strict=False)",
    "model.load_state_dict(load_state_dict(config.SDXL_CKPT), strict=False)\n        import gc, torch\n        gc.collect()\n        torch.cuda.empty_cache()"
)

# 4. Clear memory after loading F weights (12-space indent)
util_content = util_content.replace(
    "model.load_state_dict(load_state_dict(config.SUPIR_CKPT_F), strict=False)",
    "model.load_state_dict(load_state_dict(config.SUPIR_CKPT_F), strict=False)\n            import gc, torch\n            gc.collect()\n            torch.cuda.empty_cache()"
)

# 5. Clear memory after loading Q weights (12-space indent)
util_content = util_content.replace(
    "model.load_state_dict(load_state_dict(config.SUPIR_CKPT_Q), strict=False)",
    "model.load_state_dict(load_state_dict(config.SUPIR_CKPT_Q), strict=False)\n            import gc, torch\n            gc.collect()\n            torch.cuda.empty_cache()"
)

with open(util_path, "w") as f:
    f.write(util_content)
print("util.py: Model fully redirected to GPU (cuda), checkpoint reading kept on CPU.")

# Patch 8: sgm/modules/diffusionmodules/util.py — support building the schedule on GPU (cuda),
# replace betas.numpy() with betas.cpu().numpy() to avoid a TypeError when creating the model directly on GPU
util_diff_path = os.path.join("sgm", "modules", "diffusionmodules", "util.py")
with open(util_diff_path, "r") as f:
    content = f.read()
content = content.replace("\r\n", "\n")

content = content.replace(
    "return betas.numpy()",
    "return betas.cpu().numpy()"
)

with open(util_diff_path, "w") as f:
    f.write(content)
print("util.py (diffusionmodules): make_beta_schedule patched for GPU support.")

print("All patches applied. Installation complete.")

In [ ]:
#@title ##**Upload images** { display-mode: "form" }
import os
import shutil
from google.colab import files
from PIL import Image

if os.path.exists("input_images"):
    shutil.rmtree("input_images")
os.makedirs("input_images", exist_ok=True)

uploaded = files.upload()

if uploaded:
    uploaded_filename = list(uploaded.keys())[0]
    file_path = os.path.join("input_images", uploaded_filename)

    shutil.move(uploaded_filename, file_path)
    display(Image.open(file_path))

In [ ]:
#@title ##**Run** { display-mode: "form" }

SUPIR_sign = "F" #@param ["Q", "F"] {type:"string"}
upscale = 3 #@param {type:"slider", min:1, max:8, step:0.5}
min_size = 1024 #@param {type:"integer"}
edm_steps = 100 #@param {type:"slider", min:10, max:100, step:1}

#@markdown ### Sharpness and fidelity settings
s_cfg = 4 #@param {type:"slider", min:1.0, max:15.0, step:0.1}
s_stage2 = 0.95 #@param {type:"slider", min:0.0, max:1.0, step:0.01}
s_stage1 = -1 #@param {type:"integer"}

#@markdown ### Memory optimization (Tiling)
decoder_tile_size = 64 #@param [64, 96, 128, 192, 256] {type:"raw"}
encoder_tile_size = 512 #@param [256, 512, 1024] {type:"raw"}

#@markdown ### Text prompts (Universal)
no_llava = True #@param {type:"boolean"}
my_positive_prompt = "hyperrealism" #@param {type:"string"}
my_negative_prompt = "blur" #@param {type:"string"}

import os
import shutil
import subprocess
import psutil
import torch

def print_memory_stats(msg=""):
    cpu_mem = psutil.virtual_memory()
    cpu_used = cpu_mem.used / (1024 ** 3)
    cpu_total = cpu_mem.total / (1024 ** 3)
    cpu_free = cpu_mem.available / (1024 ** 3)

    print(f"\n==========================================")
    print(f"📊 {msg}")
    print(f"==========================================")
    print(f"🖥️  CPU RAM: Used {cpu_used:.2f} GB / Total {cpu_total:.2f} GB (Free: {cpu_free:.2f} GB)")
    if torch.cuda.is_available():
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        gpu_allocated = torch.cuda.memory_allocated() / (1024 ** 3)
        gpu_cached = torch.cuda.memory_reserved() / (1024 ** 3)
        gpu_free = free_bytes / (1024 ** 3)
        gpu_total = total_bytes / (1024 ** 3)
        print(f"📟 GPU VRAM: Allocated {gpu_allocated:.2f} GB / Cached {gpu_cached:.2f} GB / Total {gpu_total:.2f} GB (Free: {gpu_free:.2f} GB)")
    else:
        print("📟 GPU: Not available")
    print("==========================================\n")

# Print memory log BEFORE starting the restoration
print_memory_stats("MEMORY LOG: BEFORE RESTORATION START")

# Check for input files
if not os.listdir("input_images"):
    raise RuntimeError("The input_images folder is empty — upload an image in the 2nd cell first.")

# Clear the results folder
if os.path.exists("output_images"):
    shutil.rmtree("output_images")
os.makedirs("output_images", exist_ok=True)

# Build the launch command
cmd = [
    "python", "-u", "test.py",
    "--img_dir", "input_images",
    "--save_dir", "output_images",
    "--SUPIR_sign", str(SUPIR_sign),
    "--upscale", str(upscale),
    "--min_size", str(min_size),
    "--edm_steps", str(edm_steps),
    "--s_cfg", str(s_cfg),
    "--s_stage2", str(s_stage2),
    "--loading_half_params",
    "--use_tile_vae",
    "--encoder_tile_size", str(encoder_tile_size),
    "--decoder_tile_size", str(decoder_tile_size),
    "--a_prompt", str(my_positive_prompt),
    "--n_prompt", str(my_negative_prompt),
]

# Add s_stage1 only if you changed it yourself in the form (made it greater than 0)
if s_stage1 > 0:
    cmd.extend(["--s_stage1", str(s_stage1)])

# Disable LLaVA if the form has it set to True
if no_llava:
    cmd.append("--no_llava")

# Launch the restoration process
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
process.wait()

print(f"\nProcess exit code: {process.returncode}")

# Print memory log AFTER the restoration completes
print_memory_stats("MEMORY LOG: AFTER RESTORATION COMPLETE")

# Check the result
result_files = os.listdir("output_images")
if not result_files:
    raise RuntimeError("No result found.")
else:
    print(f"Done. {len(result_files)} result file(s) saved in output_images. Run the 'Visualize' cell to view them.")

In [ ]:
#@title ##**Visualize** { display-mode: "form" }
import os
import PIL.Image
import numpy as np
from IPython.display import HTML, display
import base64
from io import BytesIO

upload_folder = "input_images"
result_folder = "output_images"

def is_image_file(filename):
    image_extensions = {'.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff'}
    return os.path.splitext(filename.lower())[1] in image_extensions

def resize_image_maintain_aspect(image, max_width, target_height=None):
    width, height = image.size
    if width > max_width:
        new_height = int(height * max_width / width)
        image = image.resize((max_width, new_height), PIL.Image.Resampling.LANCZOS)
    if target_height is not None and image.size[1] != target_height:
        new_width = int(image.size[0] * target_height / image.size[1])
        image = image.resize((new_width, target_height), PIL.Image.Resampling.LANCZOS)
    return image

def image_to_base64(image):
    buffered = BytesIO()
    image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode('utf-8')

visualization_method = "Slider" #@param ["Side-by-Side", "Slider"]

filenames_upload = sorted([f for f in os.listdir(upload_folder) if is_image_file(f)])
filenames_upload_output = sorted([f for f in os.listdir(result_folder) if is_image_file(f)])

if not filenames_upload or not filenames_upload_output:
    print(f"Error: No images found in {upload_folder} or {result_folder}.")
else:
    for filename, filename_output in zip(filenames_upload, filenames_upload_output):
        try:
            image_original = PIL.Image.open(os.path.join(upload_folder, filename))
            image_restore = PIL.Image.open(os.path.join(result_folder, filename_output))

            if visualization_method == "Side-by-Side":
                max_width = 500
                image_original = resize_image_maintain_aspect(image_original, max_width)
                image_restore = resize_image_maintain_aspect(image_restore, max_width)
                target_height = min(image_original.size[1], image_restore.size[1])
                image_original = resize_image_maintain_aspect(image_original, max_width, target_height)
                image_restore = resize_image_maintain_aspect(image_restore, max_width, target_height)

                combined_width = image_original.size[0] + image_restore.size[0]
                combined_image = PIL.Image.new('RGB', (combined_width, target_height))
                combined_image.paste(image_original, (0, 0))
                combined_image.paste(image_restore, (image_original.size[0], 0))
                display(combined_image)

            else:
                max_width = min(image_restore.size[0], 1000)
                image_restore = resize_image_maintain_aspect(image_restore, max_width)
                target_height = image_restore.size[1]
                image_original = resize_image_maintain_aspect(image_original, max_width, target_height)

                if image_original.mode != 'RGB':
                    image_original = image_original.convert('RGB')
                if image_restore.mode != 'RGB':
                    image_restore = image_restore.convert('RGB')

                original_base64 = image_to_base64(image_original)
                restore_base64 = image_to_base64(image_restore)

                html_code = f"""
                <div style="position: relative; width: {image_restore.size[0]}px; height: {image_restore.size[1]}px; margin-bottom: 20px;">
                    <div style="position: relative; width: 100%; height: 100%; overflow: hidden;">
                        <img src="data:image/png;base64,{original_base64}" style="position: absolute; width: 100%; height: 100%;">
                        <div style="position: absolute; width: 100%; height: 100%; overflow: hidden; clip-path: inset(0 0 0 50%);">
                            <img src="data:image/png;base64,{restore_base64}" style="position: absolute; width: 100%; height: 100%;">
                        </div>
                    </div>
                    <div class="slider" style="position: absolute; top: 0; bottom: 0; width: 4px; background: white; cursor: ew-resize; left: 50%; box-shadow: 0 0 5px rgba(0,0,0,0.5);">
                        <div style="position: absolute; top: 50%; transform: translateY(-50%); width: 20px; height: 20px; background: white; border-radius: 50%; left: -8px;"></div>
                    </div>
                </div>
                <script>
                    document.querySelectorAll('.slider').forEach(slider => {{
                        let isDragging = false;
                        const container = slider.parentElement.querySelector('div:nth-child(1)');
                        const clipDiv = container.querySelector('div');

                        slider.addEventListener('mousedown', (e) => {{
                            isDragging = true;
                            e.preventDefault();
                        }});

                        document.addEventListener('mouseup', () => {{
                            isDragging = false;
                        }});

                        document.addEventListener('mousemove', (e) => {{
                            if (!isDragging) return;
                            const rect = container.getBoundingClientRect();
                            let x = e.clientX - rect.left;
                            if (x < 0) x = 0;
                            if (x > rect.width) x = rect.width;
                            const percentage = (x / rect.width) * 100;
                            slider.style.left = percentage + '%';
                            clipDiv.style.clipPath = `inset(0 0 0 ${{percentage}}%)`;
                        }});

                        slider.addEventListener('touchstart', (e) => {{
                            isDragging = true;
                            e.preventDefault();
                        }});

                        document.addEventListener('touchend', () => {{
                            isDragging = false;
                        }});

                        document.addEventListener('touchmove', (e) => {{
                            if (!isDragging) return;
                            const rect = container.getBoundingClientRect();
                            let x = e.touches[0].clientX - rect.left;
                            if (x < 0) x = 0;
                            if (x > rect.width) x = rect.width;
                            const percentage = (x / rect.width) * 100;
                            slider.style.left = percentage + '%';
                            clipDiv.style.clipPath = `inset(0 0 0 ${{percentage}}%)`;
                        }});
                    }});
                </script>
                """

                display(HTML(html_code))

        except Exception as e:
            print(f"Error processing {filename} and {filename_output}: {e}")

In [ ]:
#@title ##**Download results** { display-mode: "form" }
import os
from google.colab import files
result_files = os.listdir("output_images")
if not result_files:
    print("No result files found. Run the restoration cell first.")
else:
    result_path = os.path.join("output_images", result_files[0])
    filename = os.path.basename(result_path)
    print(f"Preparing to download file {filename}...")
    files.download(result_path)